> # Loading Libraries

use stream lit for ui

In [15]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool,InjectedToolArg
from langchain_core.messages import HumanMessage
import requests
from dotenv import load_dotenv
from typing import Annotated
load_dotenv()

True

> ### Tool create

In [16]:
@tool 

def get_conversion_factor(base_currency:str,target_currency:str) -> float:
    """
        This function fetches the currency conversion factor between a given base currency and a target currency  
    """

    url = f"https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}"
    response = requests.get(url)
    return response.json()

In [17]:
@tool

def convert(base_currency_value:int,conversion_rate:Annotated[float,InjectedToolArg])->float:
    """
        Given a currency conversion rate this func calculate the target currency value from a given base currency value
    """
    return base_currency_value*conversion_rate

In [18]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1776988801,
 'time_last_update_utc': 'Fri, 24 Apr 2026 00:00:01 +0000',
 'time_next_update_unix': 1777075201,
 'time_next_update_utc': 'Sat, 25 Apr 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.1701}

In [19]:
convert.invoke({'base_currency_value':10,'conversion_rate':94.1701})

941.701

> ### Binding

In [20]:
llm = ChatGroq(model_name="llama-3.3-70b-versatile")

In [21]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [22]:
messages = [HumanMessage("What is the conversion factor between USD and  INR and based on that can you convert 10 usd to inr")]

In [23]:
messages

[HumanMessage(content='What is the conversion factor between USD and  INR and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={})]

In [24]:
ai_message = llm_with_tools.invoke(messages)

In [25]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'ny0gtkbhc',
  'type': 'tool_call'}]

In [28]:
messages.append(ai_message)

In [29]:
import json

for tool_call in ai_message.tool_calls:
    # print(tool_call)
    # execute first tool and get value of conversion rate
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1= get_conversion_factor.invoke(tool_call)
        # print(tool_message1)
        # fetch this conversion rate
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        # append this message to message list
        messages.append(tool_message1)
    
    # execute second tool using conversion rate for tool 1
    if tool_call['name'] == 'convert':
        # fetch the current argument
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message2= convert.invoke(tool_call)
        messages.append(tool_message2)

In [30]:
messages

[HumanMessage(content='What is the conversion factor between USD and  INR and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1776988801, "time_last_update_utc": "Fri, 24 Apr 2026 00:00:01 +0000", "time_next_update_unix": 1777075201, "time_next_update_utc": "Sat, 25 Apr 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 94.1701}', name='get_conversion_factor', tool_call_id='ny0gtkbhc'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ny0gtkbhc', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 350, 'total_tokens': 416, 'completion_time': 0.193781323, 'com

In [32]:
llm.invoke(messages).content

'The conversion factor between USD and INR is 1 USD = 94.1701 INR.\n\nTo convert 10 USD to INR, we can multiply 10 by the conversion factor:\n\n10 USD * 94.1701 INR/USD = 941.701 INR\n\nSo, 10 USD is equal to approximately 941.701 INR.'